## 1. Setup

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/conjuring92/wiki-stem-corpus/wiki_stem_corpus.csv


In [2]:
!pip install -q -U transformers huggingface_hub accelerate bitsandbytes wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [3]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import wandb

print("torch:", torch.__version__)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

torch: 2.10.0+cu128
GPU  : Tesla T4


In [4]:
BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge/"

train_df = pd.read_csv(BASE + "train.csv")
test_df = pd.read_csv(BASE + "test.csv")
sample_sub = pd.read_csv(BASE + "sample_submission.csv")

options = ["A", "B", "C", "D", "E"]
print("train:", train_df.shape, "| test:", test_df.shape)
print("submission columns:", list(sample_sub.columns))

train: (2000, 8) | test: (500, 7)
submission columns: ['ID', 'Prediction']


In [5]:
PREFIXES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Identify the correct statement:",
    "Determine the correct option:",
    "Choose the correct answer:",
    "Which of the following is correct?",
]

SUFFIXES = [
    "among the listed options.",
    "from the following choices.",
    "carefully.",
    "based on the given context.",
    "among the list of options.",
]


def clean_prompt(text):
    text = str(text).strip()
    for p in PREFIXES:
        if text.startswith(p):
            text = text[len(p):].strip()
            break
    for s in SUFFIXES:
        if text.endswith(s):
            text = text[:-len(s)].strip()
            break
    return text


train_df["question"] = train_df["prompt"].apply(clean_prompt)
test_df["question"] = test_df["prompt"].apply(clean_prompt)

print("filler removed from",
      round(100 * (train_df["question"] != train_df["prompt"].str.strip()).mean(), 1),
      "% of train rows")

filler removed from 90.0 % of train rows


In [6]:
def make_key(row):
    parts = [str(row["prompt"]).strip()]
    for o in options:
        parts.append(str(row[o]).strip())
    return " || ".join(parts)


train_df["key"] = train_df.apply(make_key, axis=1)

unique_keys = list(train_df["key"].unique())
rng = np.random.RandomState(42)
rng.shuffle(unique_keys)

n_val = int(0.15 * len(unique_keys))
val_keys = set(unique_keys[:n_val])

val_df = train_df[train_df["key"].isin(val_keys)].reset_index(drop=True)
rest_df = train_df[~train_df["key"].isin(val_keys)].reset_index(drop=True)

print("unique questions:", len(unique_keys))
print("validation rows :", len(val_df))
print("remaining rows  :", len(rest_df))
print("on both sides   :", len(set(val_df["key"]) & set(rest_df["key"])), "(must be 0)")

unique questions: 1817
validation rows : 294
remaining rows  : 1706
on both sides   : 0 (must be 0)


In [7]:
val_df = val_df.sample(n=150, random_state=42).reset_index(drop=True)
print("using", len(val_df), "validation rows to save GPU time")

using 150 validation rows to save GPU time


## 4. Metrics

In [8]:
def apk(actual, predicted):
    for i in range(min(3, len(predicted))):
        if predicted[i] == actual:
            return 1.0 / (i + 1)
    return 0.0


def mapk(actual_list, predicted_list):
    total = 0.0
    for a, p in zip(actual_list, predicted_list):
        total = total + apk(a, p)
    return total / len(actual_list)


def evaluate(true_letters, ranked_lists):
    top1 = [r[0] for r in ranked_lists]
    accuracy = np.mean([t == p for t, p in zip(true_letters, top1)])
    macro_f1 = f1_score(true_letters, top1, average="macro",
                        labels=options, zero_division=0)
    map3 = mapk(list(true_letters), ranked_lists)
    return accuracy, macro_f1, map3


# tests
print(apk("A", ["A", "B", "C"]), apk("A", ["B", "A", "C"]),
      apk("A", ["B", "C", "A"]), apk("A", ["B", "C", "D"]))

1.0 0.5 0.3333333333333333 0.0


In [9]:
def rank_by_length(row):
    pairs = []
    for o in options:
        pairs.append((len(str(row[o])), o))
    pairs.sort(reverse=True)
    return [o for length, o in pairs]


length_ranked = [rank_by_length(val_df.iloc[i]) for i in range(len(val_df))]
len_acc, len_f1, len_map3 = evaluate(val_df["answer"].tolist(), length_ranked)

print("LENGTH BASELINE  acc %.4f  f1 %.4f  map3 %.4f" % (len_acc, len_f1, len_map3))
print("random guessing  map3 %.4f" % ((1 + 0.5 + 1/3) / 5))

LENGTH BASELINE  acc 0.4667  f1 0.4576  map3 0.6067
random guessing  map3 0.3667


In [10]:
def build_text(row, perm):
    lines = []
    lines.append("Question: " + str(row["question"]))
    lines.append("")
    for slot in range(5):
        original = perm[slot]
        lines.append(options[slot] + ". " + str(row[options[original]]).strip())
    lines.append("")
    lines.append("Answer with the single letter of the correct option.")
    return "\n".join(lines)


# check the reordering is really happening
demo_perm = [2, 3, 4, 0, 1]
demo = build_text(val_df.iloc[0], demo_perm)
print("Option displayed as A:")
print("  ", demo.split("\n")[2][:80])
print("Original option C was:")
print("  ", str(val_df.iloc[0]["C"])[:80])

Option displayed as A:
   A. The Heisenberg uncertainty principle states that the total angular momentum o
Original option C was:
   The Heisenberg uncertainty principle states that the total angular momentum of a


In [11]:
MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# I put the padding on the LEFT. This matters: I read the logits at the very last
# position, and with left padding the last position is a real token for every
# sequence in the batch, not a padding token.
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

print("model loaded")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model loaded


In [12]:
letter_ids = {}
for o in options:
    ids = tokenizer.encode(o, add_special_tokens=False)
    print("letter", o, "-> token ids", ids)
    letter_ids[o] = ids[0]

letter_id_list = [letter_ids[o] for o in options]
print("token ids in order:", letter_id_list)

letter A -> token ids [32]
letter B -> token ids [33]
letter C -> token ids [34]
letter D -> token ids [35]
letter E -> token ids [36]
token ids in order: [32, 33, 34, 35, 36]


In [13]:
import itertools

N_PERMS = 10
SUB_BATCH = 5   # process this many permutations per forward pass instead of
                # all N_PERMS at once -- cuts peak memory roughly in half

_all_perms = list(itertools.permutations(range(5)))
_perm_rng = np.random.RandomState(0)
_perm_rng.shuffle(_all_perms)
PERM_LIST = [list(p) for p in _all_perms[:N_PERMS]]

print("using", len(PERM_LIST), "distinct permutations out of 120 possible, sub-batch", SUB_BATCH)


def get_option_scores(row):
    totals = np.zeros(5)

    for start in range(0, len(PERM_LIST), SUB_BATCH):
        batch_perms = PERM_LIST[start:start + SUB_BATCH]
        prompts = []
        for perm in batch_perms:
            text = build_text(row, perm)
            messages = [{"role": "user", "content": text}]
            prompt = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            prompts.append(prompt)

        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        last_logits = outputs.logits[:, -1, :]
        letter_logits = last_logits[:, letter_id_list]
        log_probs = torch.log_softmax(letter_logits.float(), dim=-1).cpu().numpy()

        for r, perm in enumerate(batch_perms):
            for slot in range(5):
                original = perm[slot]
                totals[original] = totals[original] + log_probs[r, slot]

        del inputs, outputs, last_logits, letter_logits
        torch.cuda.empty_cache()

    return totals / len(PERM_LIST)


def rank_from_scores(scores):
    return [options[i] for i in np.argsort(-scores)]


using 10 distinct permutations out of 120 possible, sub-batch 5


In [14]:
# sanity check on 3 questions before running everything
for i in range(3):
    row = val_df.iloc[i]
    scores = get_option_scores(row)
    print("Q:", row["question"][:80])
    for o, s in zip(options, scores):
        print("   ", o, round(float(s), 4))
    print("  ranking:", rank_from_scores(scores), "| correct:", row["answer"])
    print("-" * 65)

Q: What is the Heisenberg uncertainty principle and how does it relate to angular m
    A -23.4936
    B -24.8186
    C -21.5061
    D -5.2311
    E -2.7311
  ranking: ['E', 'D', 'C', 'A', 'B'] | correct: E
-----------------------------------------------------------------
Q: What is the Einstein@Home project?
    A -14.8251
    B -26.4126
    C -0.0001
    D -23.0126
    E -23.1501
  ranking: ['C', 'A', 'D', 'E', 'B'] | correct: C
-----------------------------------------------------------------
Q: What are the constituents of cold dark matter?
    A 0.0
    B -30.7875
    C -30.4
    D -30.625
    E -30.7625
  ranking: ['A', 'C', 'D', 'E', 'B'] | correct: A
-----------------------------------------------------------------


## 9. Validation

In [15]:
val_ranked = []

for i in tqdm(range(len(val_df))):
    scores = get_option_scores(val_df.iloc[i])
    val_ranked.append(rank_from_scores(scores))

print("done:", len(val_ranked))

  0%|          | 0/150 [00:00<?, ?it/s]

done: 150


In [16]:
acc, f1, map3 = evaluate(val_df["answer"].tolist(), val_ranked)

print("QWEN2.5-14B + CYCLIC AVERAGING (validation)")
print("  accuracy : %.4f" % acc)
print("  macro F1 : %.4f" % f1)
print("  MAP@3    : %.4f" % map3)
print()
print("length baseline MAP@3 : %.4f" % len_map3)
print("improvement over it   : %.4f" % (map3 - len_map3))

QWEN2.5-14B + CYCLIC AVERAGING (validation)
  accuracy : 0.9133
  macro F1 : 0.9148
  MAP@3    : 0.9544

length baseline MAP@3 : 0.6067
improvement over it   : 0.3478


In [17]:
predicted_top1 = [r[0] for r in val_ranked]

comparison = pd.DataFrame({
    "true": pd.Series(val_df["answer"]).value_counts().sort_index(),
    "predicted": pd.Series(predicted_top1).value_counts().sort_index(),
})
comparison["difference"] = comparison["predicted"] - comparison["true"]
print(comparison)
print()
print("total absolute imbalance:", int(comparison["difference"].abs().sum()))
print("(version 1 with no debiasing had an imbalance of 60)")

   true  predicted  difference
A    23         24           1
B    34         29          -5
C    38         37          -1
D    30         34           4
E    25         26           1

total absolute imbalance: 12
(version 1 with no debiasing had an imbalance of 60)


In [18]:
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))
# print("logged in to W&B")

In [19]:
# WANDB_PROJECT = "<your-rollno>-t22026"

# wandb.init(project=WANDB_PROJECT, name="length-baseline",
#            config={"method": "sort options by character length",
#                    "trainable_params": 0})
# wandb.log({"val_accuracy": len_acc, "val_macro_f1": len_f1, "val_map3": len_map3})
# wandb.finish()

# wandb.init(project=WANDB_PROJECT, name="qwen2.5-14b-zeroshot-cyclic",
#            config={"model": MODEL_NAME,
#                    "method": "zero-shot letter logits + cyclic option rotation",
#                    "n_perms": N_PERMS,
#                    "quantization": "4-bit nf4",
#                    "trainable_params": 0,
#                    "val_rows": len(val_df)})
# wandb.log({"val_accuracy": acc, "val_macro_f1": f1, "val_map3": map3})
# wandb.finish()

# print("logged 2 runs")

## 11. Test predictions and submission

In [20]:
import os

CKPT_PATH = "test_ranked_ckpt.npy"

if os.path.exists(CKPT_PATH):
    test_ranked = list(np.load(CKPT_PATH, allow_pickle=True))
    start_i = len(test_ranked)
    print("resuming from checkpoint, already have", start_i, "answers")
else:
    test_ranked = []
    start_i = 0

for i in tqdm(range(start_i, len(test_df))):
    scores = get_option_scores(test_df.iloc[i])
    test_ranked.append(rank_from_scores(scores))
    if (i + 1) % 20 == 0:
        np.save(CKPT_PATH, np.array(test_ranked, dtype=object))

np.save(CKPT_PATH, np.array(test_ranked, dtype=object))
print("done:", len(test_ranked))


  0%|          | 0/500 [00:00<?, ?it/s]

done: 500


In [21]:
predictions = [" ".join(r[:3]) for r in test_ranked]

submission = pd.DataFrame({
    sample_sub.columns[0]: test_df["id"],
    sample_sub.columns[1]: predictions,
})

print("rows        :", len(submission))
print("columns     :", list(submission.columns), "| expected:", list(sample_sub.columns))
print("duplicate id:", submission[sample_sub.columns[0]].duplicated().sum())

all_good = True
for p in submission[sample_sub.columns[1]]:
    letters = p.split()
    if len(letters) != 3 or len(set(letters)) != 3:
        all_good = False
    for l in letters:
        if l not in options:
            all_good = False
print("all rows valid:", all_good)

submission.to_csv("submission.csv", index=False)
print("saved submission.csv")
submission.head(10)

rows        : 500
columns     : ['ID', 'Prediction'] | expected: ['ID', 'Prediction']
duplicate id: 0
all rows valid: True
saved submission.csv


,ID,Prediction
0,1,A D E
1,2,B E A
2,3,C B E
3,4,E A C
4,5,C D A
5,6,D A E
6,7,E A C
7,8,B E A
8,9,C D A
9,10,B D A
